# Reviewer figure: drifting gratings and depth selectivity

Inputs: `precompute_data/sftf_grating_cells.py` (`grating_tuning_cells.csv`, trial caches) and the frame-level
data and `base` fits of `precompute_data/sftf_running_models.py` (`running_models/`). Numbers are printed below and
exported to `stats/generated/stats_fig_reviewer_gratings.yaml`.

- **A**: head-fixed mouse on a wheel under two-photon imaging, drifting sinusoidal gratings on the monitors (no optic
  flow coupled to running).
- **B**: depth tuning of an example depth-selective neuron (closed-loop depth protocol; mean ± SEM, Gaussian fit).
- **C**: mean responses of the same neuron to each SF × TF, for the 8 drift directions placed around a polar plot of
  the maximum response per drift direction.
- **D**: the iscell neurons of the 6 sessions, one square per neuron: depth selective only (magenta), SF/TF tuned
  only (blue), both (the two overlapping at alpha 0.5) or neither (grey), with the number and percentage of each;
  overlap test in the section below.
- **E**: direction selectivity implied by the SF/TF fit, (pref − null) / (pref + null) =
  dsi · tanh κ / (2 − dsi), of SF/TF-tuned cells, depth selective (black outline) or not (grey); difference of the medians within each session (mean over
  sessions weighted by their number of cells), p from permuting the depth labels within sessions.
- **F**: preferred drift direction (0°: towards higher azimuth, i.e. front to back, the optic flow of forward
  running, red arrow; 90°: up) of the SF/TF-tuned cells with an implied DSI above `DSI_THR` (0.3: preferred response
  at least 1.86 times the null); χ² over 8 bins.
- **G–J**: preferred grating speed (TF0 / SF0), TF and SF against preferred virtual depth (SF/TF-tuned,
  depth-selective cells), and preferred SF against preferred optic flow speed (SF/TF- and RS/OF-tuned cells).
  Spearman ρ, 95% CI and p from a hierarchical bootstrap over sessions, Holm-corrected over the 5 pairs tested:
  grating speed × depth (G), then TF and SF × depth (H, I) to see which drives it, then SF × optic flow (J) and
  running speed (not shown); regression line (log-log, 95% band) where the Holm p < 0.05.
- **K, L**: TF (K) and SF (L) tuning curves of two example depth-selective neurons, one whose response is scaled by
  running at an unchanged preferred TF (left) and one whose preferred TF shifts with running (right), for the frames
  below 2.8 cm/s and 5 running-speed bins (the treadmill bins of figure 3, the last open-ended; darkest to lightest,
  viridis for TF, cividis for SF); mean ± SEM across presentations, SF (TF) / direction pairs from the fitted tuning.
- **M–P**: population TF (M, O) and SF (N, P) curves per running-speed bin and for the frames below 2.8 cm/s,
  depth-selective cells, relative to each cell's peak (octaves; peak from all frames, whatever the running speed, of the
  other half of the recordings), within 3 octaves of the peak (further offsets only contain the cells
  peaking at the edges of the tested range); M, N: dF/F divided by the maximum of the cell's trial-level SF/TF fit over the tested
  stimuli; O, P: the population mean curve of each bin min-max normalised. Mean ± 95%
  bootstrap CI across cells.


In [ ]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path

import flexiznam as flz
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy import stats
from scipy.stats import t as t_dist
from statsmodels.stats.multitest import multipletests

import v1_depth_map
from cottage_analysis.plotting import depth_selectivity_plots, style
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT
from v1_depth_map.figure_utils import sftf
from v1_depth_map.paths import get_figures_roots

style.setup_figure_fonts()

PROJECT = "hey2_3d-vision_foodres_20220101"
N_BOOTS = 20000  # hierarchical bootstrap over sessions of the correlations (G-J)
N_PERM = 20000  # E: permutations of the depth labels within sessions
RELOAD = False  # rebuild the caches of the depth example (B) and of the held-out speed curves (M-P)
# F: implied DSI, (pref - null) / (pref + null); 0.3 = preferred response at least 1.86 times the null
DSI_THR = 0.3
ALPHA = 0.05  # G-J: regression line where the Holm-corrected p is below this
EXAMPLE_SESSION, EXAMPLE_ROI = (
    "PZAH10.2d_S20230922",
    558,
)  # B, C: preferred depth 0.70 m
EXAMPLE_CELLS_SPEED = [  # K, L: depth-selective cells, gain only (left) and TF shift (right)
    ("PZAH10.2f_S20230914", 54, "Cell 1"),
    ("PZAH10.2f_S20230924", 95, "Cell 2"),
]
# A: vector schematic (from the Illustrator figure) and its raster preview for the axes and the PNG
SCHEMATIC = Path(v1_depth_map.__file__).parent / "figures" / "schematic_gratings.pdf"

flm = flz.get_flexilims_session(PROJECT)
_, SAVE_ROOT = get_figures_roots(flm, fig_subdir="fig_reviewer_gratings")
CACHE = SAVE_ROOT / "cache"
FRAMES_DIR = SAVE_ROOT / "running_models" / "frames"
SLOW_DIR = SAVE_ROOT / "running_models" / "frames_slow"
print(f"Saving to {SAVE_ROOT}")

In [ ]:
# cells, selections and direction selectivity implied by the SF/TF fit
cells = pd.read_csv(SAVE_ROOT / "grating_tuning_cells.csv")
kappa = np.exp(cells.fit_log_kappa)
cells["implied_dsi"] = (
    cells.fit_dsi * np.tanh(kappa) / (2 - cells.fit_dsi)
)  # (pref - null) / (pref + null)
cells["drift_dir"] = sftf.drift_direction(cells.fit_dir)

tuned = cells[cells.sftf_tuned & cells.depth_tuned]  # E, F, G-I
tuned_other = cells[cells.sftf_tuned & ~cells.depth_tuned]  # E, F (grey)
dir_tuned = tuned[tuned.implied_dsi > DSI_THR]  # E
dir_tuned_other = tuned_other[tuned_other.implied_dsi > DSI_THR]  # F (grey)
# E: labels permuted within sessions, which differ in their fraction of depth-selective cells and in DSI
sftf_cells = cells[cells.sftf_tuned]
dsi_test = sftf.stratified_median_difference(
    sftf_cells.implied_dsi, sftf_cells.depth_tuned, sftf_cells.session, n_perm=N_PERM
)
# preferred drift direction in 8 bins centred on the stimulus directions
dir_table = np.array(
    [
        np.bincount(((d.drift_dir + 22.5) % 360 // 45).astype(int), minlength=8)
        for d in (dir_tuned, dir_tuned_other)
    ]
)
dir_test = stats.chi2_contingency(dir_table)
print(
    f"E: implied DSI, medians {tuned.implied_dsi.median():.3f} (depth, n = {len(tuned)}) / "
    f"{tuned_other.implied_dsi.median():.3f} (other, n = {len(tuned_other)}); "
    f"within-session difference {dsi_test['diff']:+.3f}, permutation p = {dsi_test['p']:.2g}"
)
print(
    f"F: implied DSI > {DSI_THR:g}: {len(dir_tuned)} depth / {len(dir_tuned_other)} other; "
    f"preferred drift direction chi2 p = {dir_test.pvalue:.2g}"
)

# G-J and SF vs running speed: Spearman rho with a hierarchical bootstrap over sessions, Holm over the tested pairs
X_VARS = {  # column: (label, transform)
    "pref_speed": ("Preferred grating\nspeed (deg/s)", lambda v: v),
    "fit_tf0": ("Preferred temporal\nfrequency (Hz)", np.exp),
    "fit_sf0": ("Preferred spatial\nfrequency (cpd)", np.exp),
}
Y_VARS = {  # column: (label, transform, cells)
    "preferred_depth": ("Preferred virtual depth (cm)", lambda v: v * 100, tuned),
    "preferred_of": (
        "Preferred optic flow (deg/s)",
        lambda v: v,
        cells[cells.sftf_tuned & cells.rsof_tuned],
    ),
    "preferred_rs": (
        "Preferred running speed (cm/s)",
        lambda v: v * 100,
        cells[cells.sftf_tuned & cells.rsof_tuned],
    ),
}
# tested pairs: grating speed vs depth (G), TF and SF vs depth to see which drives it (H, I), then SF vs optic
# flow (J) and running speed (not shown)
PAIRS = [
    ("pref_speed", "preferred_depth"),
    ("fit_tf0", "preferred_depth"),
    ("fit_sf0", "preferred_depth"),
    ("fit_sf0", "preferred_of"),
    ("fit_sf0", "preferred_rs"),
]
corr = {}
for xcol, ycol in PAIRS:
    (_, xf), (_, yf, sub) = X_VARS[xcol], Y_VARS[ycol]
    corr[(xcol, ycol)] = sftf.bootstrap_spearman(
        xf(sub[xcol]), yf(sub[ycol]), sub.session, n_boots=N_BOOTS
    )
p_holm = multipletests([c["p"] for c in corr.values()], method="holm")[1]
for c, ph in zip(corr.values(), p_holm):
    c["p_holm"] = float(ph)
for (xcol, ycol), c in corr.items():
    ci = c["ci"]
    print(
        f"{xcol} vs {ycol}: n = {c['n']} ({c['n_sessions']} sessions, {c['n_mice']} mice), "
        f"rho = {c['rho']:.2f} [{ci[0]:.2f}, {ci[1]:.2f}], p = {c['p']:.4g}, p_holm = {c['p_holm']:.4g}"
    )

# B, C: example neuron; K-P: running-speed curves (held-out population curves cached by build_speed_curves)
depth_example = sftf.load_depth_example(
    CACHE, EXAMPLE_SESSION, EXAMPLE_ROI, rebuild=RELOAD
)
example_means, example_angles, sf_range, tf_range = sftf.example_condition_means(
    CACHE, EXAMPLE_SESSION, EXAMPLE_ROI
)
speed_curves = sftf.build_speed_curves(SAVE_ROOT, rebuild=RELOAD)
heldout_means, speed_labels = (
    speed_curves["heldout_means"],
    speed_curves["speed_labels"],
)
print(f"B, C example: {EXAMPLE_SESSION} ROI {EXAMPLE_ROI}")
print(
    "K, L examples:",
    ", ".join(f"{lbl}: {s} ROI {r}" for s, r, lbl in EXAMPLE_CELLS_SPEED),
)
print("M-P cells (n, sessions, mice):", speed_curves["heldout_counts"])
example_binned = {}
for session, roi, _ in EXAMPLE_CELLS_SPEED:
    d, ds = (
        np.load(FRAMES_DIR / f"{session}.npz"),
        np.load(SLOW_DIR / f"{session}.npz"),
    )
    col = int(np.flatnonzero(d["rois"] == roi)[0])
    assert ds["rois"][col] == roi
    slow = sftf.speed_binned_responses(
        ds["X"],
        ds["dff"][:, col],
        np.nan,
        recording=ds["recording"],
        single_bin_label=sftf.SLOW_LABEL,
    )
    running = sftf.speed_binned_responses(
        d["X"],
        d["dff"][:, col],
        float(d["log_rs_median"]),
        edges_cm_s=sftf.SPEED_EDGES_CM_S,
        recording=d["recording"],
    )
    example_binned[(session, roi)] = slow + running

In [ ]:
FIG_W_CM, FIG_H_CM = 18.0, 19.0
# slow frames (first colour) and running-speed bins, TF and SF
SPEED_COLORS = {
    "tf": plt.get_cmap("viridis")(np.linspace(0, 0.9, 6)),
    "sf": plt.get_cmap("cividis")(np.linspace(0, 0.9, 6)),
}


def panel_label(x, y, letter):
    fig.text(x, y, letter, fontweight="bold", fontsize=FONTSIZE_DICT["panel"], va="bottom")


def fit_line_with_ci(x, y, n_points=100):
    # least-squares line in log-log space and its 95% confidence band
    x, y = np.asarray(x, float), np.asarray(y, float)
    valid = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    lx, ly = np.log(x[valid]), np.log(y[valid])
    n = len(lx)
    p = np.polyfit(lx, ly, 1)
    lx_grid = np.linspace(lx.min(), lx.max(), n_points)
    s_err = np.sqrt(np.sum((ly - np.polyval(p, lx)) ** 2) / (n - 2))
    se_line = s_err * np.sqrt(
        1 / n + (lx_grid - lx.mean()) ** 2 / np.sum((lx - lx.mean()) ** 2)
    )
    t_val = t_dist.ppf(0.975, df=n - 2)
    ly_fit = np.polyval(p, lx_grid)
    return (
        np.exp(lx_grid),
        np.exp(ly_fit),
        np.exp(ly_fit - t_val * se_line),
        np.exp(ly_fit + t_val * se_line),
    )


fig = plt.figure(figsize=(FIG_W_CM * CM, FIG_H_CM * CM))
y_row1, h_row1 = 14.5 / FIG_H_CM, 3.8 / FIG_H_CM
y_row2, h_row2 = 9.8 / FIG_H_CM, 3.0 / FIG_H_CM
y_row3, h_row3 = 5.2 / FIG_H_CM, 2.8 / FIG_H_CM
y_row4, h_row4 = 1.1 / FIG_H_CM, 2.8 / FIG_H_CM
top1 = y_row1 + h_row1 + 0.015

# A: schematic
x_a, w_a, h_a = 0.035, 2.5 / FIG_W_CM, 1.55 / FIG_H_CM
ax = fig.add_axes([x_a, y_row1 + h_row1 - h_a, w_a, h_a])
schematic = ax.imshow(Image.open(SCHEMATIC.with_suffix(".png")))
ax.axis("off")
panel_label(x_a - 0.02, top1, "A")

# B: example depth tuning
y_b, w_b, h_b = y_row1 + 0.35 / FIG_H_CM, 2.8 / FIG_W_CM, 1.55 / FIG_H_CM
ax = fig.add_axes([x_a, y_b, w_b, h_b])
depth_selectivity_plots.plot_depth_tuning_curve(
    neurons_df=depth_example["neurons_df"],
    trials_df=depth_example["trials_depth"],
    roi=depth_example["roi"],
    rs_thr=None,
    plot_fit=True,
    plot_smooth=False,
    linewidth=1.2,
    linecolor="#2a78d6",
    closed_loop=1,
    fontsize_dict=FONTSIZE_DICT,
    markersize=2.5,
    marker="o",
    markeredgecolor="#2a78d6",
    ax=ax,
)
panel_label(x_a - 0.02, y_b + h_b + 0.015, "B")

# C: example SF x TF responses per drift direction
x_c, w_c = 0.225, 3.3 / FIG_W_CM
sftf.plot_sftf_3x3_grid(
    fig,
    (x_c, y_row1, w_c, h_row1),
    example_means,
    sftf.drift_direction(example_angles),
    sf_range,
    tf_range,
    vmax=float(np.round(np.nanmax(example_means), 1)),
    fontsize_dict=FONTSIZE_DICT,
)
panel_label(x_c - 0.025, top1, "C")

# D: depth selectivity and SF/TF tuning, one square per neuron, above E: implied direction selectivity
x_d, w_d, h_d = 0.52, 3.3 / FIG_W_CM, 1.65 / FIG_H_CM
y_d = y_row1 + (h_row1 - 2 * h_d) / 2
y_mat, h_mat = y_d + h_d + 0.2 / FIG_H_CM, 1.375 / FIG_H_CM
sftf.plot_overlap_matrix(
    fig.add_axes([x_d, y_mat, 1.1 * w_d, h_mat]),
    cells.depth_tuned,
    cells.sftf_tuned,
    legend_ax=fig.add_axes([x_d, y_mat + h_mat + 0.05 / FIG_H_CM, w_d, 0.28 / FIG_H_CM]),
    fontsize_dict=FONTSIZE_DICT,
)
panel_label(x_d - 0.03, top1, "D")
ax = fig.add_axes([x_d, y_d, w_d, h_d])
dsi_bins = np.linspace(0, 1, 21)
for d, kw in [
    (tuned_other, dict(color="lightgrey", label="other")),
    (
        tuned,
        dict(
            histtype="step",
            color="k",
            lw=1.5,
            label="depth selective",
        ),
    ),
]:
    ax.hist(d.implied_dsi, bins=dsi_bins, weights=np.full(len(d), 1 / len(d)), **kw)
ax.set_xlabel("Direction selectivity index", fontsize=FONTSIZE_DICT["label"])
ax.set_ylabel("Proportion\nof neurons", fontsize=FONTSIZE_DICT["label"])
ax.tick_params(labelsize=FONTSIZE_DICT["tick"])
ax.legend(fontsize=FONTSIZE_DICT["legend"], frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
panel_label(x_d - 0.03, y_d + h_d + 0.004, "E")

# F: preferred drift direction of direction-selective cells
x_e, w_e, h_e = 0.770, 3.0 / FIG_W_CM, 3.0 / FIG_H_CM
ax = fig.add_axes(
    [x_e, y_row1 + (h_row1 - h_e) / 2 - 0.01, w_e, h_e], projection="polar"
)
sftf.plot_polar_direction_hist(
    ax, dir_tuned_other.drift_dir.values, color="lightgrey", fontsize_dict=FONTSIZE_DICT
)
sftf.plot_polar_direction_hist(
    ax, dir_tuned.drift_dir.values, step=True, color="k", lw=1.5, fontsize_dict=FONTSIZE_DICT
)
rmax = ax.get_ylim()[1]
ax.annotate(
    "",
    xy=(0, 0.9 * rmax),
    xytext=(0, 0),
    arrowprops=dict(arrowstyle="->", color="crimson", lw=1.3, mutation_scale=10),
    zorder=5,
)
ax.text(
    np.deg2rad(-25),
    0.7 * rmax,
    "Optic flow",
    fontsize=FONTSIZE_DICT["legend"],
    color="crimson",
    ha="center",
    va="center",
    zorder=6,
)
panel_label(x_e - 0.075, top1, "F")

# G-J: preferred grating speed, TF and SF vs depth, SF vs optic flow
SCATTERS = [
    ("G", "pref_speed", "preferred_depth"),
    ("H", "fit_tf0", "preferred_depth"),
    ("I", "fit_sf0", "preferred_depth"),
    ("J", "fit_sf0", "preferred_of"),
]
w_sc, gap_sc, x_sc = 3.0 / FIG_W_CM, 1.4 / FIG_W_CM, 1.0 / FIG_W_CM
for k, (letter, xcol, ycol) in enumerate(SCATTERS):
    (xlabel, xf), (ylabel, yf, sub), st = X_VARS[xcol], Y_VARS[ycol], corr[(xcol, ycol)]
    x, y = xf(sub[xcol]), yf(sub[ycol])
    x0 = x_sc + k * (w_sc + gap_sc)
    ax = fig.add_axes([x0, y_row2, w_sc, h_row2])
    ax.scatter(x, y, s=5, color="k", alpha=0.35, linewidths=0, zorder=2)
    if st["p_holm"] < ALPHA:
        xg, yg, lo, hi = fit_line_with_ci(x, y)
        ax.fill_between(xg, lo, hi, color="crimson", alpha=0.2, lw=0, zorder=3)
        ax.plot(xg, yg, color="crimson", lw=1.2, zorder=4)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(xlabel, fontsize=FONTSIZE_DICT["label"])
    ax.set_ylabel(ylabel, fontsize=FONTSIZE_DICT["label"], labelpad=1)
    ax.tick_params(labelsize=FONTSIZE_DICT["tick"])
    ax.spines[["top", "right"]].set_visible(False)
    panel_label(x0 - 0.045, y_row2 + h_row2 + 0.015, letter)


# K, L: example cells, TF and SF curves per running-speed bin and for the slow frames
def plot_curves(ax, curves, variable, ylabel):
    freq = sftf.FREQ_COLS[variable]
    colors = dict(zip([sftf.SLOW_LABEL, *speed_labels], SPEED_COLORS[variable]))
    for label, color in colors.items():
        c = curves[curves["bin"] == label]
        ax.errorbar(
            c[freq],
            c["mean"],
            yerr=c["sem"],
            color=color,
            lw=1.1,
            marker="o",
            ms=2.5,
            label=label,
        )
    ax.set_xscale("log")
    vals = np.sort(curves[freq].unique())
    ax.set_xticks(vals, [f"{v:g}" for v in vals])
    ax.minorticks_off()
    ax.set_ylabel(ylabel, fontsize=FONTSIZE_DICT["label"])
    ax.tick_params(labelsize=FONTSIZE_DICT["tick"])
    ax.spines[["top", "right"]].set_visible(False)


w_ex, gap_ex, x_ex = 1.95 / FIG_W_CM, 0.45 / FIG_W_CM, 0.050
cells_idx = cells.set_index(["session", "roi"])
for letter, variable, y_row, xlabel in [
    ("K", "tf", y_row3, "Temporal frequency (Hz)"),
    ("L", "sf", y_row4, "Spatial frequency (cpd)"),
]:
    panel_label(x_ex - 0.030, y_row + h_row3 + 0.015, letter)
    for j, (session, roi, label) in enumerate(EXAMPLE_CELLS_SPEED):
        ax = fig.add_axes([x_ex + j * (w_ex + gap_ex), y_row, w_ex, h_row3])
        curves, _ = sftf.tf_curves_by_speed(
            example_binned[(session, roi)],
            threshold=sftf.TF_SELECT_THRESHOLD,
            variable=variable,
            popt=cells_idx.loc[(session, roi)],
        )
        plot_curves(ax, curves, variable, "ΔF/F" if j == 0 else "")
        if variable == "sf":
            ax.tick_params(axis="x", rotation=45)
    fig.text(
        x_ex + w_ex + gap_ex / 2,
        y_row - 0.045,
        xlabel,
        ha="center",
        fontsize=FONTSIZE_DICT["label"],
    )
    # legend next to the example cells
    ax = fig.add_axes([0.315, y_row, 2.1 / FIG_W_CM, h_row3])
    ax.axis("off")
    ax.legend(
        handles=[
            plt.Line2D([0], [0], color=c, marker="o", ms=3.6, lw=1.1, label=lbl)
            for lbl, c in zip([sftf.SLOW_LABEL, *speed_labels], SPEED_COLORS[variable])
        ],
        fontsize=FONTSIZE_DICT["legend"],
        frameon=False,
        loc="center left",
    )

# M-P: held-out population curves, depth-selective cells, within MAX_DELTA octaves of the peak (further
# offsets only contain cells peaking at the edges of the tested range)
MAX_DELTA = 3
w_pop = 3.35 / FIG_W_CM
for letter, variable, own_peak, x0, y_row in [
    ("M", "tf", False, 0.495, y_row3),
    ("N", "sf", False, 0.495, y_row4),
    ("O", "tf", True, 0.755, y_row3),
    ("P", "sf", True, 0.755, y_row4),
]:
    ax = fig.add_axes([x0, y_row, w_pop, h_row3])
    sftf.plot_heldout_speed_curves_panel(
        ax,
        variable,
        heldout_means[heldout_means.delta.abs() <= MAX_DELTA],
        speed_curves["mean_peaks"],
        speed_labels,
        colors=SPEED_COLORS[variable][1:],
        slow_color=SPEED_COLORS[variable][0],
        norm_to_own_peak=own_peak,
        fontsize_dict=FONTSIZE_DICT,
    )
    name = variable.upper()
    ax.set_ylabel(
        "Normalized response" if own_peak else "Response / fitted peak",
        fontsize=FONTSIZE_DICT["label"],
    )
    ax.set_xlabel(f"{name} relative to peak (octaves)", fontsize=FONTSIZE_DICT["label"])
    ax.set_xticks(np.arange(-MAX_DELTA, MAX_DELTA + 1))
    panel_label(x0 - 0.035, y_row + h_row3 + 0.015, letter)

# PDF with the vector schematic in A, PNG and SVG with its raster preview
sftf.save_with_vector_image(
    fig,
    schematic,
    SCHEMATIC,
    SAVE_ROOT / "fig_sftf_analysis.pdf",
    png_path=SAVE_ROOT / "fig_sftf_analysis.png",
    svg_path=SAVE_ROOT / "fig_sftf_analysis.svg",
)

## Overlap of depth-selective and SF/TF-tuned cells (panel D)

Are depth selectivity and SF/TF tuning carried by separate sub-populations? The count of cells that are both is
compared with the count expected if the two were independent. The fractions of tuned cells differ between sessions,
so the expected count is computed in each session (`n_depth * n_sftf / n`) and summed. The test is a Mantel-Haenszel
test of the pooled odds ratio over the per-session 2 x 2 tables (depth selective x SF/TF tuned).

In [ ]:
# overlap of depth-selective and SF/TF-tuned cells, expected count per session, Mantel-Haenszel test
from statsmodels.stats.contingency_tables import StratifiedTable

overlap_rows, overlap_tables = [], []
for session, sc in cells.groupby("session"):
    n, n_depth, n_sftf = len(sc), int(sc.depth_tuned.sum()), int(sc.sftf_tuned.sum())
    overlap_rows.append(
        dict(
            session=session,
            n=n,
            n_depth=n_depth,
            n_sftf=n_sftf,
            n_both=int((sc.depth_tuned & sc.sftf_tuned).sum()),
            n_both_expected=n_depth * n_sftf / n,
        )
    )
    overlap_tables.append(
        pd.crosstab(sc.depth_tuned, sc.sftf_tuned)
        .reindex(index=[True, False], columns=[True, False], fill_value=0)
        .to_numpy()
    )
overlap_sessions = pd.DataFrame(overlap_rows)
overlap = overlap_sessions.drop(columns="session").sum()
overlap_cmh = StratifiedTable(overlap_tables)
overlap_or = overlap_cmh.oddsratio_pooled
overlap_p = overlap_cmh.test_null_odds().pvalue

print(overlap_sessions.round(1).to_string(index=False))
print(
    f"Depth selective and SF/TF tuned: {overlap.n_both:.0f} / {overlap.n:.0f} cells "
    f"({100 * overlap.n_both / overlap.n:.1f}%); expected if independent within each session: "
    f"{overlap.n_both_expected:.1f} ({100 * overlap.n_both_expected / overlap.n:.1f}%)"
)
print(f"Mantel-Haenszel pooled odds ratio {overlap_or:.2f}, p = {overlap_p:.2g}")

## Export statistics

Every value is read from a variable computed above; nothing is recomputed. Written to
`v1_depth_map/stats/generated/stats_fig_reviewer_gratings.yaml` (group `reviewer`);
`python -m v1_depth_map.stats.build_manuscript_stats` compiles it into the dashboard and the LaTeX macros.

In [ ]:
from v1_depth_map.stats import export_figure_stats


def _fmt_pval(p):
    # p-value as it should read in LaTeX, in math mode so it works in text or math
    p = float(p)
    return "$p < 0.0001$" if p < 1e-4 else f"$p = {p:.3g}$"


def _stat(raw, formatted, macro, description):
    return dict(
        raw=raw, formatted=formatted, latex_macro=macro, description=description
    )


def _count(n, macro, description):
    return _stat(int(n), f"{int(n):,}", macro, description)


def _pct(fraction, macro, description):
    return _stat(float(100 * fraction), f"{100 * fraction:.0f}\\%", macro, description)


stats_reviewer = {
    "n_neurons": _count(
        overlap.n, "statRevGratNeurons", "iscell neurons in the 6 grating sessions"
    ),
    "n_sessions": _count(
        cells.session.nunique(),
        "statRevGratSessions",
        "Sessions with drifting gratings",
    ),
    "n_mice": _count(
        cells.session.str.split("_").str[0].nunique(),
        "statRevGratMice",
        "Mice with drifting gratings",
    ),
    "n_depth": _count(
        overlap.n_depth, "statRevGratDepthNeurons", "Depth-selective neurons"
    ),
    "rsof_threshold": _stat(
        float(cells.rsof_threshold.iloc[0]),
        f"{cells.rsof_threshold.iloc[0]:.4f}",
        "statRevGratRsofThreshold",
        "RS/OF held-out R² threshold of figure 2 (95th percentile of the empirical null, iscell ROIs of all sessions)",
    ),
    "n_rsof": _count(
        cells.rsof_tuned.sum(),
        "statRevGratRsofNeurons",
        "RS/OF-tuned neurons (figure 2 threshold) in the grating sessions",
    ),
    "pct_depth": _pct(
        overlap.n_depth / overlap.n, "statRevGratPctDepth", "Percentage depth selective"
    ),
    "n_sftf": _count(
        overlap.n_sftf, "statRevGratSftfNeurons", "SF/TF-tuned neurons, cross-validated"
    ),
    "pct_sftf": _pct(
        overlap.n_sftf / overlap.n, "statRevGratPctSftf", "Percentage SF/TF tuned"
    ),
    "n_both": _count(
        overlap.n_both, "statRevGratBothN", "Depth-selective and SF/TF-tuned neurons"
    ),
    "pct_both": _pct(
        overlap.n_both / overlap.n,
        "statRevGratPctBoth",
        "Percentage depth selective and SF/TF tuned",
    ),
    "n_both_expected": _stat(
        float(overlap.n_both_expected),
        f"{overlap.n_both_expected:.0f}",
        "statRevGratBothExpectedN",
        "Depth selective and SF/TF tuned expected if independent, computed per session and summed",
    ),
    "pct_both_expected": _pct(
        overlap.n_both_expected / overlap.n,
        "statRevGratPctBothExpected",
        "Percentage both expected if independent, computed per session",
    ),
    "overlap_or": _stat(
        float(overlap_or),
        f"{overlap_or:.2f}",
        "statRevGratOverlapOR",
        "Mantel-Haenszel pooled odds ratio over sessions, depth selective x SF/TF tuned",
    ),
    "overlap_p": _stat(
        float(overlap_p),
        _fmt_pval(overlap_p),
        "statRevGratOverlapP",
        "Mantel-Haenszel test of the pooled odds ratio over sessions",
    ),
}
CORR_MACROS = {  # (x, y): macro stem, description; the tested PAIRS
    ("pref_speed", "preferred_depth"): (
        "SpeedDepth",
        "preferred grating speed vs preferred depth (G)",
    ),
    ("fit_tf0", "preferred_depth"): ("TfDepth", "preferred TF vs preferred depth (H)"),
    ("fit_sf0", "preferred_depth"): ("SfDepth", "preferred SF vs preferred depth (I)"),
    ("fit_sf0", "preferred_of"): ("SfOf", "preferred SF vs preferred optic flow (J)"),
    ("fit_sf0", "preferred_rs"): (
        "SfRs",
        "preferred SF vs preferred running speed (not shown)",
    ),
}
for (xcol, ycol), (stem, description) in CORR_MACROS.items():
    st = corr[(xcol, ycol)]
    key = stem[0].lower() + stem[1:]
    stats_reviewer[f"{key}_n"] = _count(
        st["n"], f"statRevGrat{stem}N", f"Neurons, {description}"
    )
    stats_reviewer[f"{key}_rho"] = _stat(
        float(st["rho"]),
        f"{st['rho']:.2f}",
        f"statRevGrat{stem}Rho",
        f"Spearman rho, {description}",
    )
    stats_reviewer[f"{key}_p"] = _stat(
        float(st["p"]),
        _fmt_pval(st["p"]),
        f"statRevGrat{stem}P",
        f"Hierarchical bootstrap (sessions) p, {description}",
    )
    stats_reviewer[f"{key}_p_holm"] = _stat(
        float(st["p_holm"]),
        _fmt_pval(st["p_holm"]),
        f"statRevGrat{stem}PHolm",
        f"Holm-corrected p over the {len(corr)} pairs, {description}",
    )
stats_reviewer["dsi_median_depth"] = _stat(
    float(tuned.implied_dsi.median()),
    f"{tuned.implied_dsi.median():.2f}",
    "statRevGratDsiMedianDepth",
    "Median implied DSI, SF/TF-tuned depth-selective neurons (E)",
)
stats_reviewer["dsi_median_other"] = _stat(
    float(tuned_other.implied_dsi.median()),
    f"{tuned_other.implied_dsi.median():.2f}",
    "statRevGratDsiMedianOther",
    "Median implied DSI, SF/TF-tuned other neurons (E)",
)
stats_reviewer["dsi_diff"] = _stat(
    dsi_test["diff"],
    f"{dsi_test['diff']:.3f}",
    "statRevGratDsiDiff",
    "Implied DSI, depth selective minus other: difference of the medians within each session, mean over "
    "sessions weighted by their number of cells (E)",
)
stats_reviewer["dsi_p"] = _stat(
    dsi_test["p"],
    _fmt_pval(dsi_test["p"]),
    "statRevGratDsiP",
    f"Permutation p of the within-session DSI difference, depth labels permuted within sessions "
    f"({N_PERM:,} permutations) (E)",
)
stats_reviewer["dir_tuned_n_depth"] = _count(
    len(dir_tuned),
    "statRevGratDirTunedN",
    f"Depth-selective neurons with implied DSI > {DSI_THR:g} (F)",
)
stats_reviewer["dir_tuned_n_other"] = _count(
    len(dir_tuned_other),
    "statRevGratDirTunedOtherN",
    f"Other neurons with implied DSI > {DSI_THR:g} (F)",
)
stats_reviewer["dir_p"] = _stat(
    float(dir_test.pvalue),
    _fmt_pval(dir_test.pvalue),
    "statRevGratDirP",
    "chi2 p, preferred drift direction (8 bins), depth selective vs other (F)",
)
for variable, panels in [("tf", "M, O"), ("sf", "N, P")]:  # M-P: depth-selective cells
    name = variable.upper()
    for count, (suffix, what) in zip(
        speed_curves["heldout_counts"][(variable, True)],
        [("N", "Depth-selective neurons"), ("Sessions", "Sessions"), ("Mice", "Mice")],
    ):
        stats_reviewer[f"speed_curves_{variable}_{suffix.lower()}"] = _count(
            count,
            f"statRevGratSpeedCurves{name.capitalize()}{suffix}",
            f"{what}, population {name} curves per running-speed bin ({panels})",
        )
export_figure_stats("fig_reviewer_gratings", stats_reviewer, figure="reviewer")
for k,v in stats_reviewer.items():
    print(f"{k}: {v['formatted']} ({v['description']})")